# Ablation Experiments 60–67 — Dean's exp 16 (commit `fe0e939`)

Trains 8 variants of Dean's exp 16 architecture (MobileNetV2 amputated at block 10, 2 shared attention blocks, `FREEZE_UP_TO_BLOCK=6`) on cleaned training data (`corrections.csv` + 366-entry `bad_images.csv`). Saved `.h5` matches Dean's `keras_version=2.15.0` so it loads on the Pi (TF 2.15) natively — no patching, no TFLite conversion required.

---

## Architecture (shared by all 8)

| key | value |
|---|---|
| `BASE_MODEL` | MobileNetV2 α=1.0 |
| `CUT_AT_BLOCK` | 10 (amputated) |
| `FREEZE_UP_TO_BLOCK` | 6 |
| `NUM_ATTN_BLOCKS` | 2 |
| `SPLIT_ATTN_AT_BLOCK` | **None** (= 2 SHARED attention blocks — exp 16 signature) |
| `USE_CLEAN_DATA` | True |
| `EPOCHS_WARMUP` / `EPOCHS_FINETUNE` | 5 / 195 |

---

## Pre-flight: upload to Drive root (`My Drive/`)

| File | Source on Mac |
|---|---|
| `picar_data.zip` | data prep output |
| `train_ablation.py` | `src/train_ablation.py` |
| `bad_images.csv` | `data/bad_images.csv` |
| `corrections.csv` | `data/corrections.csv` |
| `requirements.txt` | repo root |

---

## Run order

1. **SETUP cell** — builds `/content/py310` venv with TF 2.15.0 + the rest of `requirements.txt`. ~5 min first time, instant on subsequent runs in the same runtime.
2. **A1 → A4** (BATCH 1, tonight)
3. **A5 → A8** (BATCH 2, tomorrow)
4. Each cell saves `<EXP_NAME>_best.h5` to `/content/drive/MyDrive/picar_models/`.

Colab Pro Standard = 1 GPU; cells run serially. ~3-5 hr per experiment.

---

## If SETUP fails

The most common failure is Colab's `ubuntugis` PPA timing out during `apt-get install`. Run the **RESET** cell (immediately below this overview) to wipe the broken venv + disable the bad PPA, then re-run SETUP.

---

## A2 reminder

A2 uses **Huber for angle + MSE for speed** — *not* BCE. The earlier huber+BCE combo caused a ModelCheckpoint pathology (best epoch saved at ~epoch 17 because BCE punished the model heavily as soon as it became confident, so val_loss minimum was at the under-trained early epoch). Huber+MSE isolates the angle change cleanly.

In [ ]:
# RESET — only run if SETUP failed and you need to start fresh.
# This wipes the broken venv and disables the PPA that times out.

import os, shutil
if os.path.exists('/content/py310'):
    shutil.rmtree('/content/py310')
    print('🗑️  Removed /content/py310')

!sudo rm -f /etc/apt/sources.list.d/ubuntugis*.list
!sudo apt-get update -q
!sudo apt-get install -y --fix-missing python3.10-venv python3.10-dev python3-setuptools-whl python3-pip-whl -q

print('✅ Reset done. Now re-run the SETUP cell below.')

## SETUP cell (run in EVERY new tab first)

Builds a Python 3.10 venv, installs `requirements.txt` + CUDA wheels, verifies TF 2.15.0 / Keras 2.15.0 / GPU. ~5 min first time per runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil
ZIP_PATH    = '/content/drive/MyDrive/picar_data.zip'
SCRIPT_PATH = '/content/drive/MyDrive/train_ablation.py'
BAD_PATH    = '/content/drive/MyDrive/bad_images.csv'
CORR_PATH   = '/content/drive/MyDrive/corrections.csv'
REQ_PATH    = '/content/drive/MyDrive/requirements.txt'

for p in [ZIP_PATH, SCRIPT_PATH, BAD_PATH, CORR_PATH, REQ_PATH]:
    assert os.path.exists(p), f'❌ Upload to Drive root: {p}'

# Unzip data
if not os.path.exists('/content/PiCar/data/training_data/training_data'):
    !mkdir -p /content/PiCar
    !cd /content/PiCar && unzip -q -o /content/drive/MyDrive/picar_data.zip
    print('✅ Unzipped data')
else:
    print('⏭️  Data already unzipped')

# Copy script + csvs + requirements
os.makedirs('/content/PiCar/src', exist_ok=True)
os.makedirs('/content/PiCar/data', exist_ok=True)
shutil.copy(SCRIPT_PATH, '/content/PiCar/src/train_ablation.py')
shutil.copy(BAD_PATH,    '/content/PiCar/data/bad_images.csv')
shutil.copy(CORR_PATH,   '/content/PiCar/data/corrections.csv')
shutil.copy(REQ_PATH,    '/content/PiCar/requirements.txt')
print('✅ Files in place')

# === Python 3.10 venv with requirements.txt (TF 2.15.0 — Pi-compatible) ===
VENV     = '/content/py310'
VENV_PY  = f'{VENV}/bin/python'
VENV_PIP = f'{VENV}/bin/pip'

if not os.path.exists(VENV_PY):
    # Disable PPAs that intermittently time out and break apt
    !sudo rm -f /etc/apt/sources.list.d/ubuntugis*.list
    !sudo rm -f /etc/apt/sources.list.d/deadsnakes*.list
    !sudo rm -f /etc/apt/sources.list.d/graphics-drivers*.list
    !sudo apt-get update -q
    !sudo apt-get install -y --fix-missing python3.10-venv python3.10-dev python3-setuptools-whl python3-pip-whl -q
    !python3.10 -m venv {VENV}
    assert os.path.exists(VENV_PY), 'venv shell missing — apt install of python3.10-venv likely failed'
    !{VENV_PIP} install -q --upgrade pip
    # numpy<2 (TF 2.15 requires it); requirements.txt installs tensorflow==2.15.0 (CPU-only wheel)
    !{VENV_PIP} install -q "numpy>=1.23.5,<2.0"
    !{VENV_PIP} install -q -r /content/PiCar/requirements.txt
    # CUDA wheels — tensorflow[and-cuda]==2.15.0 would have installed these, but its
    # tensorrt-libs==8.6.1 is no longer on PyPI. Install the CUDA libs individually
    # (skip tensorrt; we don't use TF-TRT).
    !{VENV_PIP} install -q \
        "nvidia-cublas-cu12==12.2.5.6" \
        "nvidia-cuda-cupti-cu12==12.2.142" \
        "nvidia-cuda-nvcc-cu12==12.2.140" \
        "nvidia-cuda-nvrtc-cu12==12.2.140" \
        "nvidia-cuda-runtime-cu12==12.2.140" \
        "nvidia-cudnn-cu12==8.9.4.25" \
        "nvidia-cufft-cu12==11.0.8.103" \
        "nvidia-curand-cu12==10.3.3.141" \
        "nvidia-cusolver-cu12==11.5.2.141" \
        "nvidia-cusparse-cu12==12.1.2.141" \
        "nvidia-nccl-cu12==2.16.5" \
        "nvidia-nvjitlink-cu12==12.2.140"
    print('✅ py310 venv built')
else:
    print('⏭️  venv already exists')

# Verify versions match Pi (use standalone keras package for version — tf.keras lazy-loads
# in TF 2.15 and doesn't expose __version__ reliably).
import subprocess
out = subprocess.run([VENV_PY, '-c',
    "import sys, tensorflow as tf, keras; "
    "print(sys.version.split()[0], tf.__version__, keras.__version__)"
], capture_output=True, text=True)
print('venv check stdout:', out.stdout.strip())
if out.returncode != 0:
    print('venv check stderr:', out.stderr.strip())
    raise RuntimeError('venv broken — see stderr above')
parts = out.stdout.strip().split()
assert parts[1] == '2.15.0', f'expected TF 2.15.0, got {parts[1]}'
assert parts[2].startswith('2.15.'), f'expected Keras 2.15.x, got {parts[2]}'
print(f'✅ venv versions: Python {parts[0]} / TF {parts[1]} / Keras {parts[2]}')

# Verify GPU visible to the venv
out_gpu = subprocess.run([VENV_PY, '-c',
    "import tensorflow as tf; gpus=tf.config.list_physical_devices('GPU'); "
    "print('GPU:', gpus if gpus else 'NO GPU DETECTED')"
], capture_output=True, text=True)
print(out_gpu.stdout.strip())
assert 'NO GPU DETECTED' not in out_gpu.stdout, \
    'GPU not detected — Runtime must be set to T4/L4/A100, and CUDA wheels must be installed'

# wandb login (system pip — venv has its own wandb from requirements.txt; netrc is shared)
!pip install wandb -q
import wandb
wandb.login()

# Drive save dir
DRIVE_SAVE_DIR = '/content/drive/MyDrive/picar_models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f'✅ Drive save dir: {DRIVE_SAVE_DIR}')

# Keep-alive (fights Colab idle)
import threading, time
from IPython.display import display, Javascript
def keep_alive():
    while True:
        time.sleep(600)
        try: display(Javascript('console.log("alive")'))
        except: pass
threading.Thread(target=keep_alive, daemon=True).start()
print('✅ SETUP done — training cells launch via /content/py310/bin/python')

## Cell A1 — `60_baseline16_clean` (BATCH 1)

Dean exp 16 (fe0e939) + corrections.csv + 366-bad-images cleanup

In [ ]:
import os, json
EXP_NAME = '60_baseline16_clean'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'Dean exp 16 (fe0e939) + corrections.csv + 366-bad-images cleanup',
    'EPOCHS_FINETUNE': 245,
    'CUT_AT_BLOCK': 10,
    'FREEZE_UP_TO_BLOCK': 6,
    'NUM_ATTN_BLOCKS': 2,
    'SPLIT_ATTN_AT_BLOCK': None,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!/content/py310/bin/python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log

import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A2 — `61_huber_angle_only` (BATCH 1)

fe0e939 + **Huber for angle, MSE for speed**. Tests Huber's robustness-to-noise effect on angle alone. Speed stays as MSE (Dean's default) — avoids BCE checkpoint pathology and isolates the angle change.

In [ ]:
import os, json
EXP_NAME = '61_huber_angle_only'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'fe0e939 + Huber for angle, MSE for speed (isolates Huber on angle alone, no BCE pathology)',
    'EPOCHS_FINETUNE': 245,
    'CUT_AT_BLOCK': 10, 'FREEZE_UP_TO_BLOCK': 6,
    'NUM_ATTN_BLOCKS': 2, 'SPLIT_ATTN_AT_BLOCK': None,
    'LOSS_FUNCTION': 'huber',           # for angle head
    'SPEED_LOSS_FUNCTION': 'mse',       # for speed head (explicit — overrides default)
    'USE_CLEAN_DATA': True,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!/content/py310/bin/python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log

import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A3 — `62_crop120_30` (BATCH 1)

fe0e939 + crop top 120 (was 110)

In [ ]:
import os, json
EXP_NAME = '62_crop120_30'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'fe0e939 + crop top 120 (was 110)',
    'EPOCHS_FINETUNE': 245,
    'CUT_AT_BLOCK': 10,
    'FREEZE_UP_TO_BLOCK': 6,
    'NUM_ATTN_BLOCKS': 2,
    'SPLIT_ATTN_AT_BLOCK': None,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'CROP_TOP_PIXELS': 120,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!/content/py310/bin/python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log

import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A4 — `63_progressive` (BATCH 1)

fe0e939 + progressive unfreezing (smoother optim)

In [ ]:
import os, json
EXP_NAME = '63_progressive'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'fe0e939 + progressive unfreezing (smoother optim)',
    'EPOCHS_FINETUNE': 245,
    'CUT_AT_BLOCK': 10,
    'FREEZE_UP_TO_BLOCK': 6,
    'NUM_ATTN_BLOCKS': 2,
    'SPLIT_ATTN_AT_BLOCK': None,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'USE_PROGRESSIVE_UNFREEZING': True,
    'PROGRESSIVE_LR_DECAY': 0.85,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!/content/py310/bin/python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log

import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A5 — `64_cutout_smaller` (BATCH 2)

fe0e939 + cutout 10/30 (smaller, was 10/50)

In [ ]:
import os, json
EXP_NAME = '64_cutout_smaller'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'fe0e939 + cutout 10/30 (smaller, was 10/50)',
    'EPOCHS_FINETUNE': 245,
    'CUT_AT_BLOCK': 10,
    'FREEZE_UP_TO_BLOCK': 6,
    'NUM_ATTN_BLOCKS': 2,
    'SPLIT_ATTN_AT_BLOCK': None,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'AUG_CUTOUT_MIN_PIX': 10,
    'AUG_CUTOUT_MAX_PIX': 30,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!/content/py310/bin/python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log

import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A6 — `65_cutout_aggressive` (BATCH 2)

fe0e939 + cutout 15/60 prob 0.7

In [ ]:
import os, json
EXP_NAME = '65_cutout_aggressive'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'fe0e939 + cutout 15/60 prob 0.7',
    'EPOCHS_FINETUNE': 245,
    'CUT_AT_BLOCK': 10,
    'FREEZE_UP_TO_BLOCK': 6,
    'NUM_ATTN_BLOCKS': 2,
    'SPLIT_ATTN_AT_BLOCK': None,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'AUG_CUTOUT_PROB': 0.7,
    'AUG_CUTOUT_MIN_PIX': 15,
    'AUG_CUTOUT_MAX_PIX': 60,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!/content/py310/bin/python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log

import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A7 — `66_dense_smaller` (BATCH 2)

fe0e939 + dense head 128→64 (was 256→128)

In [ ]:
import os, json
EXP_NAME = '66_dense_smaller'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'fe0e939 + dense head 128→64 (was 256→128)',
    'EPOCHS_FINETUNE': 245,
    'CUT_AT_BLOCK': 10,
    'FREEZE_UP_TO_BLOCK': 6,
    'NUM_ATTN_BLOCKS': 2,
    'SPLIT_ATTN_AT_BLOCK': None,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'DENSE_UNITS_1': 128,
    'DENSE_UNITS_2': 64,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!/content/py310/bin/python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log

import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')

## Cell A8 — `67_bn_locked` (BATCH 2)

fe0e939 + explicit BatchNorm lock

In [ ]:
import os, json
EXP_NAME = '67_bn_locked'
OVERRIDES = {
    'EXPERIMENT_NAME': EXP_NAME,
    'DESCRIPTION': 'fe0e939 + explicit BatchNorm lock',
    'EPOCHS_FINETUNE': 245,
    'CUT_AT_BLOCK': 10,
    'FREEZE_UP_TO_BLOCK': 6,
    'NUM_ATTN_BLOCKS': 2,
    'SPLIT_ATTN_AT_BLOCK': None,
    'LOSS_FUNCTION': 'mse',
    'USE_CLEAN_DATA': True,
    'LOCK_BATCHNORM': True,
}
os.environ['CONFIG_OVERRIDES'] = json.dumps(OVERRIDES)
%cd /content/PiCar
!/content/py310/bin/python src/train_ablation.py 2>&1 | tee /content/{EXP_NAME}.log

import shutil
src = f'/content/PiCar/experiments/{EXP_NAME}/best_model.h5'
dst = f'/content/drive/MyDrive/picar_models/{EXP_NAME}_best.h5'
if os.path.exists(src): shutil.copy(src, dst); print(f'✅ {dst}')